
## Kiểm chứng không rò rỉ dữ liệu (Anti-Leakage Verification)

### Mục tiêu:
1. **Kiểm tra rò rỉ group**: Khẳng định giao tập `group_id` giữa `train`, `val`, `test` là rỗng.
2. **Kiểm tra tương đồng pHash**: Khẳng định không có cặp ảnh nào có khoảng cách Hamming pHash <= 5 nằm ở hai split khác nhau.
3. **Thống kê phân bố lớp**: Tính số lượng bounding box của 6 lớp lỗi theo từng split.
4. **Thống kê phân bố kích thước lỗi**: Tính tỷ lệ các hộp bounding box `small`, `medium`, `large` theo từng split.
5. **Xuất báo cáo**: Ghi toàn bộ kết quả vào `results/leakage_report.txt` theo đúng quy chuẩn.

In [1]:
from datetime import datetime, timezone
import os
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import yaml

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for parent in [current, *current.parents]:
        if (parent / "configs").is_dir() and (parent / "data").is_dir():
            return parent
    return current

ROOT = find_project_root()
print(f"Project root: {ROOT}")


Project root: /Users/Project/kltn-pcb


In [2]:
CONFIG_PATH = ROOT / "configs/base.yaml"
GROUPS_PATH = ROOT / "data/splits/groups.csv"
SPLITS_DIR = ROOT / "data/splits"
TRAIN_TXT = SPLITS_DIR / "train.txt"
VAL_TXT = SPLITS_DIR / "val.txt"
TEST_TXT = SPLITS_DIR / "test.txt"
BOXES_CSV = ROOT / "results/size_distribution_boxes.csv"
REPORT_PATH = ROOT / "results/leakage_report.txt"
PAIRS_CSV = ROOT / "results/pku_near_duplicate_pairs.csv"

# Đọc configs/base.yaml
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    base_config = yaml.safe_load(f)

CLASSES = [c.strip().lower().replace(" ", "_") for c in base_config.get("classes", [])]
print(f"Cấu hình 6 classes: {CLASSES}")

# 1. Đọc các file split
splits_map = {}
split_counts = {}
for split_name, txt_file in [("train", TRAIN_TXT), ("val", VAL_TXT), ("test", TEST_TXT)]:
    if not txt_file.is_file():
        raise FileNotFoundError(f"Không tìm thấy file split: {txt_file}")
    with open(txt_file, "r", encoding="utf-8") as f:
        paths = [line.strip() for line in f if line.strip()]
    for p in paths:
        splits_map[p] = split_name
    split_counts[split_name] = len(paths)

print(f"Đã nạp {len(splits_map):,} đường dẫn ảnh:")
for s, count in split_counts.items():
    print(f"  - {s:<5}: {count:>5,} ảnh ({count/len(splits_map):.2%})")


Cấu hình 6 classes: ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']
Đã nạp 4,837 đường dẫn ảnh:
  - train: 3,989 ảnh (82.47%)
  - val  :   368 ảnh (7.61%)
  - test :   480 ảnh (9.92%)


In [3]:
# 2. Đọc groups.csv và map group_id, phash cho từng ảnh
df_groups = pd.read_csv(GROUPS_PATH, dtype={"group_id": str})
df_groups["split"] = df_groups["image_path"].map(splits_map)

# Chỉ giữ lại các ảnh nằm trong các split đang xét
df_split_groups = df_groups.dropna(subset=["split"]).copy()
print(f"Ánh xạ thành công {len(df_split_groups):,} ảnh vào nhóm và pHash.")

# 3. Kiểm tra rò rỉ group_id (không group nào nằm ở hơn 1 split)
groups_by_split = {
    s: set(df_split_groups[df_split_groups["split"] == s]["group_id"].unique())
    for s in ["train", "val", "test"]
}

overlap_tr_val = groups_by_split["train"] & groups_by_split["val"]
overlap_tr_te = groups_by_split["train"] & groups_by_split["test"]
overlap_va_te = groups_by_split["val"] & groups_by_split["test"]

has_group_leak = bool(overlap_tr_val or overlap_tr_te or overlap_va_te)

if has_group_leak:
    group_status = f"[FAIL] Phát hiện rò rỉ group_id giữa các split: train-val={overlap_tr_val}, train-test={overlap_tr_te}, val-test={overlap_va_te}"
else:
    group_status = "[PASS] Giao của tập group giữa train/val/test = rỗng"

print(group_status)
for s in ["train", "val", "test"]:
    print(f"  - {s.capitalize():<5} groups ({len(groups_by_split[s])}): {sorted(groups_by_split[s])}")


Ánh xạ thành công 4,837 ảnh vào nhóm và pHash.
[PASS] Giao của tập group giữa train/val/test = rỗng
  - Train groups (6): ['01', '04', '05', '06', '07', '12']
  - Val   groups (2): ['10', '11']
  - Test  groups (2): ['08', '09']


In [4]:
# 4. Kiểm tra cặp phash distance <= 5 nằm ở hai split khác nhau
leaking_phash_pairs = []

if PAIRS_CSV.is_file():
    # Tận dụng kết quả đã tính từ kết quả audit toàn bộ ảnh
    df_pairs = pd.read_csv(PAIRS_CSV)
    for _, row in df_pairs.iterrows():
        s_a = splits_map.get(row["image_path_a"])
        s_b = splits_map.get(row["image_path_b"])
        if s_a and s_b and s_a != s_b and row["hamming_distance"] <= 5:
            leaking_phash_pairs.append({
                "image_a": row["image_path_a"],
                "image_b": row["image_path_b"],
                "split_a": s_a,
                "split_b": s_b,
                "distance": row["hamming_distance"],
            })
else:
    # Tính trực tiếp bằng vectorization numpy nếu chưa có file pairs
    phash_records = df_split_groups[["image_path", "split", "phash"]].to_dict("records")
    by_split = {"train": [], "val": [], "test": []}
    for r in phash_records:
        by_split[r["split"]].append((r["image_path"], int(r["phash"], 16)))
    for s1, s2 in [("train", "val"), ("train", "test"), ("val", "test")]:
        for img_a, h_a in by_split[s1]:
            for img_b, h_b in by_split[s2]:
                dist = (h_a ^ h_b).bit_count()
                if dist <= 5:
                    leaking_phash_pairs.append({
                        "image_a": img_a,
                        "image_b": img_b,
                        "split_a": s1,
                        "split_b": s2,
                        "distance": dist,
                    })

has_phash_leak = len(leaking_phash_pairs) > 0

if has_phash_leak:
    phash_status = f"[FAIL] Phát hiện {len(leaking_phash_pairs)} cặp phash distance <= 5 nằm ở hai split khác nhau"
else:
    phash_status = "[PASS] Không có cặp phash distance <= 5 nằm ở hai split khác nhau"

print(phash_status)
if has_phash_leak:
    print("Chi tiết các cặp vi phạm đầu tiên:")
    for p in leaking_phash_pairs[:10]:
        print(f"  {p['image_a']} ({p['split_a']}) <--> {p['image_b']} ({p['split_b']}), dist={p['distance']}")


[PASS] Không có cặp phash distance <= 5 nằm ở hai split khác nhau


In [5]:
# 5 & 6. Tính phân bố 6 lớp và kích thước size_bin theo split
df_boxes = pd.read_csv(BOXES_CSV)
raw_prefix = "data/raw/PKU-Market-PCB(Data enhanced version)/"
df_boxes["full_image_path"] = raw_prefix + df_boxes["image_path"]
df_boxes["split"] = df_boxes["full_image_path"].map(splits_map)

# Chuẩn hóa tên lớp về chữ thường nhất quán
df_boxes["clean_class"] = df_boxes["class_name"].str.strip().str.lower().str.replace(" ", "_")

# Tạo bảng chéo (cross-tab) số box theo 6 lớp
class_table = pd.crosstab(df_boxes["clean_class"], df_boxes["split"], margins=True)
cols_order = [c for c in ["train", "val", "test", "All"] if c in class_table.columns]
class_table = class_table.reindex(columns=cols_order)

# Tạo bảng chéo kích thước size_bin (small, medium, large)
size_table = pd.crosstab(df_boxes["size_bin"], df_boxes["split"], margins=True)
size_table = size_table.reindex(columns=cols_order)

# Tỷ lệ % size bin theo từng split
size_pct = pd.crosstab(df_boxes["size_bin"], df_boxes["split"], normalize="columns") * 100
size_pct = size_pct.reindex(columns=[c for c in ["train", "val", "test"] if c in size_pct.columns])

print("=== BẢNG PHÂN BỐ 6 LỚP THEO SPLIT ===")
print(class_table.to_string())
print("\n=== BẢNG PHÂN BỐ KÍCH THƯỚC THEO SPLIT ===")
print(size_table.to_string())


=== BẢNG PHÂN BỐ 6 LỚP THEO SPLIT ===
split            train   val  test    All
clean_class                              
missing_hole      2598   300   408   3306
mouse_bite        2570   304   400   3274
open_circuit      2410   324   404   3138
short             2405   304   396   3105
spur              2338   300   400   3038
spurious_copper   2410   328   404   3142
All              14731  1860  2412  19003

=== BẢNG PHÂN BỐ KÍCH THƯỚC THEO SPLIT ===
split     train   val  test    All
size_bin                          
medium      329   290    12    631
small     14402  1570  2400  18372
All       14731  1860  2412  19003


In [6]:
# 7 & 8. Tổng hợp nội dung báo cáo và xuất ra results/leakage_report.txt
now_iso = datetime.now(timezone.utc).isoformat()

report_lines = []
report_lines.append("=" * 80)
report_lines.append("BÁO CÁO KIỂM CHỨNG DATA LEAKAGE (TUẦN 4 - BƯỚC 3)")
report_lines.append("=" * 80)
report_lines.append(f"Thời gian kiểm chứng: {now_iso}")
report_lines.append(f"Tổng số ảnh kiểm tra: {len(splits_map):,} (Train: {split_counts['train']:,}, Val: {split_counts['val']:,}, Test: {split_counts['test']:,})")
report_lines.append(f"Tổng số groups      : {len(df_split_groups['group_id'].unique())} (Train: {len(groups_by_split['train'])}, Val: {len(groups_by_split['val'])}, Test: {len(groups_by_split['test'])})")
report_lines.append("")
report_lines.append(group_status)
report_lines.append(f"  - Train groups: {sorted(groups_by_split['train'])}")
report_lines.append(f"  - Val groups  : {sorted(groups_by_split['val'])}")
report_lines.append(f"  - Test groups : {sorted(groups_by_split['test'])}")
report_lines.append("")
report_lines.append(phash_status)
if has_phash_leak:
    report_lines.append(f"  - Số cặp vi phạm: {len(leaking_phash_pairs)}")
    for p in leaking_phash_pairs[:10]:
        report_lines.append(f"    * {p['image_a']} ({p['split_a']}) <-> {p['image_b']} ({p['split_b']}), dist={p['distance']}")
report_lines.append("")
report_lines.append("[INFO] Phân bố 6 lớp theo split:")
report_lines.append("-" * 80)
header_cls = f"{'Tên lớp lỗi':<18} | {'Train':>10} | {'Val':>10} | {'Test':>10} | {'Tổng cộng':>10}"
report_lines.append(header_cls)
report_lines.append("-" * 80)
for cls in CLASSES:
    tr_c = class_table.loc[cls, "train"] if cls in class_table.index and "train" in class_table.columns else 0
    va_c = class_table.loc[cls, "val"] if cls in class_table.index and "val" in class_table.columns else 0
    te_c = class_table.loc[cls, "test"] if cls in class_table.index and "test" in class_table.columns else 0
    tot_c = class_table.loc[cls, "All"] if cls in class_table.index and "All" in class_table.columns else (tr_c + va_c + te_c)
    report_lines.append(f"{cls:<18} | {tr_c:>10,} | {va_c:>10,} | {te_c:>10,} | {tot_c:>10,}")
report_lines.append("-" * 80)
tot_tr = class_table.loc["All", "train"]
tot_va = class_table.loc["All", "val"]
tot_te = class_table.loc["All", "test"]
tot_all = class_table.loc["All", "All"]
report_lines.append(f"{'Tổng số box':<18} | {tot_tr:>10,} | {tot_va:>10,} | {tot_te:>10,} | {tot_all:>10,}")
report_lines.append("-" * 80)
report_lines.append("")
report_lines.append("[INFO] Tỷ lệ box small/medium/large theo split:")
report_lines.append("-" * 80)
header_size = f"{'Kích thước bin':<20} | {'Train':>16} | {'Val':>16} | {'Test':>16} | {'Tổng cộng':>16}"
report_lines.append(header_size)
report_lines.append("-" * 80)
for sz in ["small", "medium", "large"]:
    tr_s = size_table.loc[sz, "train"] if sz in size_table.index and "train" in size_table.columns else 0
    va_s = size_table.loc[sz, "val"] if sz in size_table.index and "val" in size_table.columns else 0
    te_s = size_table.loc[sz, "test"] if sz in size_table.index and "test" in size_table.columns else 0
    tot_s = size_table.loc[sz, "All"] if sz in size_table.index and "All" in size_table.columns else (tr_s + va_s + te_s)
    
    tr_p = size_pct.loc[sz, "train"] if sz in size_pct.index and "train" in size_pct.columns else 0.0
    va_p = size_pct.loc[sz, "val"] if sz in size_pct.index and "val" in size_pct.columns else 0.0
    te_p = size_pct.loc[sz, "test"] if sz in size_pct.index and "test" in size_pct.columns else 0.0
    tot_p = (tot_s / tot_all * 100) if tot_all > 0 else 0.0
    
    report_lines.append(f"{sz:<20} | {tr_s:>7,} ({tr_p:5.2f}%) | {va_s:>7,} ({va_p:5.2f}%) | {te_s:>7,} ({te_p:5.2f}%) | {tot_s:>7,} ({tot_p:5.2f}%)")
report_lines.append("-" * 80)
report_lines.append(f"{'Tổng số box':<20} | {tot_tr:>16,} | {tot_va:>16,} | {tot_te:>16,} | {tot_all:>16,}")
report_lines.append("-" * 80)
report_lines.append("")
if not has_group_leak and not has_phash_leak:
    report_lines.append("KẾT LUẬN: TOÀN BỘ CÁC BÀI KIỂM THỬ KHÔNG RÒ RỈ DỮ LIỆU ĐỀU ĐẠT CHUẨN [PASS].")
else:
    report_lines.append("KẾT LUẬN: CÓ BÀI KIỂM THỬ KHÔNG ĐẠT [FAIL]. CẦN ĐIỀU CHỈNH LẠI PHÂN VÙNG.")
report_lines.append("=" * 80)

final_report_text = "\n".join(report_lines) + "\n"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(final_report_text, encoding="utf-8")
print(f"Đã lưu báo cáo vào: {REPORT_PATH.relative_to(ROOT)}\n")
print(final_report_text)

# Nếu có lỗi rò rỉ dữ liệu, exit với code != 0
if has_group_leak or has_phash_leak:
    raise RuntimeError("Kiểm tra rò rỉ thất bại: Có lỗi leakage!")


Đã lưu báo cáo vào: results/leakage_report.txt

BÁO CÁO KIỂM CHỨNG DATA LEAKAGE (TUẦN 4 - BƯỚC 3)
Thời gian kiểm chứng: 2026-09-19T11:46:30.434050+00:00
Tổng số ảnh kiểm tra: 4,837 (Train: 3,989, Val: 368, Test: 480)
Tổng số groups      : 10 (Train: 6, Val: 2, Test: 2)

[PASS] Giao của tập group giữa train/val/test = rỗng
  - Train groups: ['01', '04', '05', '06', '07', '12']
  - Val groups  : ['10', '11']
  - Test groups : ['08', '09']

[PASS] Không có cặp phash distance <= 5 nằm ở hai split khác nhau

[INFO] Phân bố 6 lớp theo split:
--------------------------------------------------------------------------------
Tên lớp lỗi        |      Train |        Val |       Test |  Tổng cộng
--------------------------------------------------------------------------------
missing_hole       |      2,598 |        300 |        408 |      3,306
mouse_bite         |      2,570 |        304 |        400 |      3,274
open_circuit       |      2,410 |        324 |        404 |      3,138
short       